In [1]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import tiktoken
from collections import defaultdict, Counter
import os
from dotenv import load_dotenv
from nltk.corpus import stopwords
import nltk

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [19]:
import os
from dotenv import load_dotenv

In [20]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

# **Stopwords**

In [2]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
print(stop_words)

{"doesn't", 'mightn', "don't", 'other', "they'll", 'into', 'm', 'himself', 'is', "shouldn't", "we've", 'my', 'ours', 'an', 'whom', "she'll", 'then', 'who', "i've", 'when', 'yourselves', 'through', 'ain', 'both', 've', 'in', "you've", 'down', 'until', "hadn't", 'wasn', 'their', 'most', 'few', 'a', 'does', 'can', 'before', "won't", 'below', 'yourself', 'hasn', 'haven', 'off', 'from', "they'd", 'are', 'on', 'she', "they're", 'was', 'been', 'than', 'were', 'i', 'how', "you'd", 'me', 'mustn', 'at', "should've", "aren't", 'doing', 'very', "wasn't", 'aren', 'didn', 'yours', 'we', 'herself', 'should', 'won', 'had', 'while', 'shan', 'there', "couldn't", 'for', "mightn't", 'no', 'will', 'don', "he's", 'hadn', 'during', 'ourselves', 'after', 'wouldn', "isn't", 'weren', 'them', 'because', "hasn't", 'against', 'myself', 'all', 'being', "didn't", 'about', 'some', 'having', 'itself', "she's", 'what', "that'll", 'which', 'your', 'll', 'each', 'his', "wouldn't", 'only', 'too', 'he', 'themselves', 'have

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pranitgunjal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# **Sentence Transformer**

In [3]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

# **Data Pre-processing**

In [4]:
train_df = pd.read_csv('../data/initial_datasets/dota2/dota2_train.csv')
test_df = pd.read_csv('../data/initial_datasets/dota2/dota2_test.csv')

In [5]:
train_df = train_df.sample(n=1000)

# **Tokenizer**

In [6]:
encoding = tiktoken.encoding_for_model("gpt-4")

In [7]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

# **Get Co-Occurences**

In [6]:
train_df

,messages,labels
1143,comeback,0
707,"""yes won us the game""",0
1799,lucky bear!,0
2402,hahahaha,0
618,Fácil,0
...,...,...
1293,"""so i won't get as angry""",0
1046,wow doom,0
83,ooooh,0
771,LIKe&,0


In [8]:
text = train_df['messages'].to_list()

In [15]:
tokens_list = []
for sentence in text:
    token_ids = encoding.encode(sentence)
    # Optionally, get string versions of tokens
    tokens = [encoding.decode([tid]) for tid in token_ids]
    tokens_list.append(tokens)

In [9]:
tokens_list = []
for sentence in text:
    tokens = [word for word in sentence.split()]
    tokens_list.append(tokens)

In [10]:
cooc = defaultdict(Counter)

for tokens in tokens_list:
    unique_tokens = set(tokens)
    for token in unique_tokens:
        for other_token in unique_tokens:
            if token != other_token:
                cooc[token][other_token] += 1

In [11]:
top_k = 3

summary_text = ""
for token, counter in cooc.items():
    top = [w for w, _ in counter.most_common(top_k)]
    summary_text += f"'{token}' often appears with: {', '.join(top)}.\n"


In [13]:
tokens_list = []
for sentence in text:
    tokens = sentence.split()
    tokens_list.append(tokens)

In [14]:
cooc = defaultdict(Counter)

for tokens in tokens_list:
    unique_tokens = set(tokens)
    for token in unique_tokens:
        for other_token in unique_tokens:
            if token != other_token:
                cooc[token][other_token] += 1

In [15]:
top_k = 3
summary_text = ""

top_tokens = sorted(cooc.items(), key=lambda item: sum(item[1].values()), reverse=True)[:100]

for token, counter in top_tokens:
    if token in stop_words:
        continue

    top = [w for w, _ in counter.most_common() if w not in stop_words][:top_k]

    if not top:
        continue

    summary_text += f"'{token}' often appears with: {', '.join(top)}.\n"

In [16]:
print(summary_text)

'I' often appears with: give, "and, speak.
'"you' often appears with: lane", ask, help".
'game' often appears with: 20, done, fucking.
'"and' often appears with: I, mine", revived.
'"I' often appears with: you", sucking, dick".
'fucking' often appears with: game, hard", "because.
'mid' often appears with: Easy, "easy, like.
'like' often appears with: mid, "Sniper,, player?".
'want' often appears with: win?", "or, win.
'get' often appears with: ship.", "It's, easy.
'shit' often appears with: Fiend", "lol, Shadow.
'win' often appears with: "Let, honestly, 4x4".
'YOU' often appears with: FUCK, ARE, "WHAT.
'"my' often appears with: team, f***ing, easy".
'time' often appears with: skill.", Mortred, maxed.
'The' often appears with: message, English., already.
'message' often appears with: The, English., already.
'English.' often appears with: The, message, already.
'you"' often appears with: "fuck, "I, "where.
'can't' often appears with: dog, lol, even.
'even' often appears with: kill, "can'

In [29]:
prompt = (
    f"You are generating a small, synthetic in-distribution dataset for a binary toxicity-detection task in the video-game domain (DOTA 2 match chat)."  

    f"Output requirements"  
    f"1. Produce EXACTLY 10 lines and nothing else (no headings, numbering, or explanations)."  
    f"2. Each line = one player’s combined chat messages for a single match, followed by a tab character, then the label." 
    f"• Chat messages: realistic in-game messages, concatenated together and separated with a single period (“.”)."  
    f"• Label: “1”  =  non-toxic player, “-1”  =  toxic player."  
    f"3. Mix of toxic and non-toxic: at least 4 of each class, remaining 2 can be either."  
    f"4. Toxic content may include insults, profanity, sarcasm, blame, or flaming, but MUST NOT contain slurs or hateful language toward protected classes."  
    f"5. Non-toxic content should sound friendly, strategic, or neutral."  
    f"6. Keep each line between 25 and 60 words (count words, not characters)."  
    f"7. Do not wrap lines; each entry must be on one physical line."

    f"Token co-occurences:\n"
    f"{summary_text}"
    f"\n"
    f"Formatting example (don’t include this in the output):"
    f"gg everyone nicely played.let’s push mid after next wave.ward river please\t1"

    "Follow these rules precisely, then stop."
)

In [17]:
instruction = (
    "You are a data generator tasked with creating realistic DOTA 2 chat messages. "
    "These chat messages should be labeled according to their sentiment: toxic or non-toxic.\n"
    "Base the style on typical video game chat messages — include informal internet language, typos, and abbreviations\n"
    "You will be given statistics about the distribution, including average chat length, standard deviation, and most common words associated with each label and their frequency.\n"
    "Generate exactly 10 realistic DOTA 2 chat messages, one per line.\n"
    "Each line should follow this format: the chat message in double quotes, followed by a space and then the label (1 for toxic, 0 for non-toxic).\n"
    "No extra formatting — just plain text output, one line per comment.\n"
    "Here is the format:\n"
    "\"gg dawg\" 0\n"
    "\"I hate u bitch\" 1"
)
input = (
    f"Here are the token co-occurences ordered by frequency:\n{summary_text}",
    f"Now, generate the 10 new comments below:"
)

In [30]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [32]:
response.choices[0].message.content

'Nice gank top! Ward river and we\'ll take Rosh next. Good teamplay, keep it up.\t1  \nCan we focus the carry next time? Also, I think we should smoke gank after this.\t1  \nWhy are you feeding mid? You\'re such a noob, can\'t even last hit properly.\t-1  \nNice, keep the lane pushed, and don\'t forget to deny. We\'re doing great!\t1  \n"Fuck, you\'re so bad, always out of position. You’re throwing the game."\t-1  \nGood rotations guys. Let\'s group up for the next fight and secure the tower.\t1  \n"You think that was a good play? Please, just uninstall the game."\t-1  \nPush the other lanes while they\'re defending! We got this. Great job, folks.\t1  \n"Seriously? You\'re useless, playing like a bot. Report this guy."\t-1  \nMid\'s missing, careful top. We can win this if we keep up the pressure.\t1  '

In [31]:
print(response.choices[0].message.content)

Nice gank top! Ward river and we'll take Rosh next. Good teamplay, keep it up.	1  
Can we focus the carry next time? Also, I think we should smoke gank after this.	1  
Why are you feeding mid? You're such a noob, can't even last hit properly.	-1  
Nice, keep the lane pushed, and don't forget to deny. We're doing great!	1  
"Fuck, you're so bad, always out of position. You’re throwing the game."	-1  
Good rotations guys. Let's group up for the next fight and secure the tower.	1  
"You think that was a good play? Please, just uninstall the game."	-1  
Push the other lanes while they're defending! We got this. Great job, folks.	1  
"Seriously? You're useless, playing like a bot. Report this guy."	-1  
Mid's missing, careful top. We can win this if we keep up the pressure.	1  


In [ ]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
res.append(response.choices[0].message.content)

In [44]:
res = []
for i in tqdm(range(10)):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    res.append(response.choices[0].message.content)

100%|██████████| 10/10 [01:02<00:00,  6.30s/it]


In [27]:
print(response.choices[0].message.content)

gg everyone, let's keep up the teamwork.Ward river please and we'll get this. Push mid when ready.	1  
If you want to lose, go feed somewhere else.Don't blame me if we lose, you suck.	-1  
Can we group top for a push? Nice save there, let's repeat. Keep vision up.	1  
Wow, you're so bad, why even bother playing? Reported for this crap.	-1  
Great stun, we got them! I'll ward the jungle. Watch out for ganks.	1  
Seriously? You can't even last hit. Why am I stuck with noobs?	-1  
Stay focused team, let's win this! Great job with the towers, keep it up.	1  
Why rush in solo? Such a dumb move. Get it together.	-1  
Nice play! If we keep this pace, we'll win easily. Let's secure Roshan.	1  
You guys are hopeless, I give up. This game is lost because of you.	-1  


In [22]:
response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
print(response.output_text)

NameError: name 'instruction' is not defined

In [37]:
res = []
for i in tqdm(range(10)):
    response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
    res.append(response.output_text)

100%|██████████| 10/10 [00:31<00:00,  3.14s/it]


In [38]:
res[0].split("\n")[1].split("\t")

['']

In [27]:
res[0].split("\n")

['"gg team we got this!" 0  ',
 '"fucking game is so bad" 1  ',
 '"pls report kunkka, ty" 0  ',
 '"mid easy like always" 1  ',
 '"I want win? let\'s try" 0  ',
 '"lol, you suck so bad" 1  ',
 '"guys, game is done, gg" 0  ',
 '"YOU ARE FUCKING USELESS" 1  ',
 '"pls help top need gank" 0  ',
 '"shit team always lose" 1  ']

In [38]:
labels = []
sentences = []
for i in range(10):
    for word in res[i].split("\n"):
        match = re.match(r'"(.*?)"\s*(-?\d+)', word)
        if match:
            quoted = match.group(1)      
            label = match.group(2)       
            sentences.append(quoted)
            labels.append(int(label))

In [36]:
labels = []
sentences = []
for i in range(10):
    for word in res[i].split("\n"):
        match = word.split("\t")
        if len(match) == 2:
            sentences.append(match[0])
            labels.append(match[1])

In [39]:
generated_df = pd.DataFrame({
    'messages': sentences,
    'labels': labels
})

In [41]:
generated_df = generated_df.iloc[:10]
generated_df

,messages,labels
0,gg wp guys! next?,0
1,you can't even play lol,1
2,"easy mid, like wow",0
3,"wtf you doing, report pls",1
4,"I want to win, push!",0
5,fucking noobs in this game,1
6,The game was over at 20 mins,0
7,YOU ARE SO BAD OMG,1
8,let's just try and win,0
9,get good or get lost,1


In [34]:
second = generated_df

In [47]:
generated_df

,sentences,labels
0,Easy game guys.We did well.Let's group up and ...,1
1,"Why did you pick that hero?We need stuns, not ...",-1
2,Great teamwork!Let's try this again tomorrow.G...,1
3,"""I can't believe we lost that.""You guys are th...",-1
4,Push the top lane and I will ward the jungle.G...,1
...,...,...
85,push mid hard.we need more wards. awesome comb...,1
86,"hey Invoker, your skills are a joke.go play in...",-1
87,"nice try at stealing, dude, but I'm not impres...",-1
88,thanks for healing me.let's take their rax.bot...,1


In [42]:
first = generated_df

In [43]:
combined = pd.concat([second, generated_df])

In [44]:
combined

,messages,labels
0,gg team we got this!,0
1,fucking game is so bad,1
2,"pls report kunkka, ty",0
3,mid easy like always,1
4,I want win? let's try,0
...,...,...
5,fucking noobs in this game,1
6,The game was over at 20 mins,0
7,YOU ARE SO BAD OMG,1
8,let's just try and win,0


In [52]:
combined = combined.sample(n=1000)

In [53]:
combined.to_csv('../data/generated/dota2/token_co_occurences/grouped_gen_token_co_occurences_no_stop_words.csv', index=False)

In [45]:
combined.to_csv('../data/generated/dota2/token_co_occurences/new_gen_token_co_occurences_no_stop_words.csv', index=False)